# Assignment 4 - Survival Analysis

This notebook walks through the three required survival analysis methods:
1. Kaplan-Meier Analysis
2. Cox Proportional Hazards Model
3. Random Survival Forest

**You can complete the assignment by:**
- Implementing functions in `students/` modules AND running this notebook, OR
- Writing a custom script that calls your implemented functions

**Both approaches are valid** as long as you generate all required outputs.

In [31]:
import sys
print(sys.executable)

c:\Users\JAY\AppData\Local\Programs\Python\Python312\python.exe


In [32]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())
print("Absolute outputs path:", Path("outputs").resolve())

Current working directory: c:\Users\JAY\Desktop\summer 2026\assignment-4-survival-analysis-Ayu223shi\notebooks
Absolute outputs path: C:\Users\JAY\Desktop\summer 2026\assignment-4-survival-analysis-Ayu223shi\notebooks\outputs


In [33]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import sys
from pathlib import Path


sys.path.append(str(Path("..").resolve()))  # Add parent directory to sys.path

# Import student implementations
from students import (
    fit_kaplan_meier,
    compute_logrank_test,
    plot_km_curves,
    fit_cox_model,
    extract_cox_summary,
    test_proportional_hazards,
    fit_random_survival_forest,
    compute_concordance_index,
    get_feature_importance,
    plot_feature_importance,
    set_plot_style,
)

# Set plotting style
set_plot_style()

# Create outputs directory
Path('outputs').mkdir(exist_ok=True)

## 1. Load Data

Load the RADCURE clinical dataset.

**Download instructions:**
- Option 1: From TCIA website (see data/README.md)
- Option 2: From Blackboard assignment folder

The file should be saved as `data/RADCURE-clinical-data.xlsx` or `.csv`

In [34]:
# Load RADCURE clinical data
# If you have the Excel file:
data = pd.read_excel('../data/RADCURE-clinical-data.xlsx')

# If you converted to CSV:
# data = pd.read_csv('../data/RADCURE-clinical-data.csv')

# Display basic info
print(f"Dataset shape: {data.shape}")
print(f"\nFirst few column names: {list(data.columns[:10])}")
print(f"\nData types:\n{data.dtypes.value_counts()}")

# Show first few rows
data.head()

Dataset shape: (3346, 34)

First few column names: ['patient_id', 'Age', 'Sex', 'ECOG PS', 'Smoking PY', 'Smoking Status', 'Ds Site', 'Subsite', 'T', 'N']

Data types:
object            22
datetime64[ns]     7
float64            3
int64              2
Name: count, dtype: int64


,patient_id,Age,Sex,ECOG PS,Smoking PY,Smoking Status,Ds Site,Subsite,T,N,...,Local,Date Local,Regional,Date Regional,Distant,Date Distant,2nd Ca,Date 2nd Ca,RADCURE-challenge,ContrastEnhanced
0,RADCURE-0005,62.6,Female,ECOG 0,50,Ex-smoker,Oropharynx,post wall,T4b,N2c,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,0
1,RADCURE-0006,87.3,Male,ECOG 2,25,Ex-smoker,Larynx,Glottis,T1b,N0,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,1
2,RADCURE-0007,49.9,Male,ECOG 1,15,Ex-smoker,Oropharynx,Tonsil,T3,N2b,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,1
3,RADCURE-0009,72.3,Male,ECOG 1,30,Ex-smoker,Unknown,NaN,T0,N2c,...,NaN,NaT,NaN,NaT,NaN,NaT,S (suspicious),2008-05-27,0,0
4,RADCURE-0010,59.7,Female,ECOG 0,0,Non-smoker,Oropharynx,Tonsillar Fossa,T4b,N0,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,0


In [35]:
# Prepare survival variables
# RADCURE column names may vary - check your file!
time_col = 'Length FU'  # Or check actual column name
event_col = 'Status'  # 1 = died, 0 = alive

# Convert Status to numeric (0 = Alive, 1 = Dead)
data[event_col] = (
    data[event_col]
    .astype(str)
    .str.strip()
    .map({
        "Alive": 0,
        "Dead": 1
    })
)

# Verify columns exist
print("Available columns related to survival:")
print([col for col in data.columns if 'time' in col.lower() or 'death' in col.lower() or 'survival' in col.lower()])

# Check event distribution
print(f"\nEvent distribution ({event_col}):")
print(data[event_col].value_counts())

print(f"\nCensoring rate: {(1 - data[event_col].mean()) * 100:.1f}%")

# Summary statistics
print("\nSurvival time (days):")
print(data[time_col].describe())

# Verify conversion
print("\nStatus dtype:", data[event_col].dtype)
print("Unique values:", data[event_col].unique())


Available columns related to survival:
['Date of Death', 'Cause of Death']

Event distribution (Status):
Status
0    2288
1    1058
Name: count, dtype: int64

Censoring rate: 68.4%

Survival time (days):
count    3346.000000
mean        4.136047
std         2.734757
min         0.156164
25%         1.879452
50%         3.663014
75%         5.791781
max        12.909589
Name: Length FU, dtype: float64

Status dtype: int64
Unique values: [1 0]


In [36]:
print(data.columns.tolist())

['patient_id', 'Age', 'Sex', 'ECOG PS', 'Smoking PY', 'Smoking Status', 'Ds Site', 'Subsite', 'T', 'N', 'M ', 'Stage', 'Path', 'HPV', 'Tx Modality', 'Chemo', 'RT Start', 'Dose', 'Fx', 'Last FU', 'Status', 'Length FU', 'Date of Death', 'Cause of Death', 'Local', 'Date Local', 'Regional', 'Date Regional', 'Distant', 'Date Distant', '2nd Ca', 'Date 2nd Ca', 'RADCURE-challenge', 'ContrastEnhanced']


## 2. Kaplan-Meier Analysis

Compare survival curves between patient groups.

In [37]:
# Create grouping variable for KM analysis
# Option 1: Age groups
data['Age_Group'] = pd.cut(data['Age'], bins=[0, 60, 100], labels=['≤60', '>60'])

# Option 2: HPV status (if available)
# group_col = 'HPV_Combined'

# Option 3: Overall stage (Early vs Advanced)
# data['Stage_Group'] = data['Overall_Stage'].map({
#     'I': 'Early', 'II': 'Early',
#     'III': 'Advanced', 'IV': 'Advanced'
# })

# Choose which grouping to use
group_col = 'Age_Group'

print(f"Grouping by: {group_col}")
print(data[group_col].value_counts())

Grouping by: Age_Group
Age_Group
>60    1932
≤60    1414
Name: count, dtype: int64


In [38]:
# Fit KM curves
km_models = fit_kaplan_meier(
    data=data,
    time_col=time_col,
    event_col=event_col,
    group_col=group_col
)

print(f"Fitted KM curves for {len(km_models)} groups: {list(km_models.keys())}")

Fitted KM curves for 2 groups: ['>60', '≤60']


In [39]:
# Compute log-rank test
logrank_results = compute_logrank_test(
    data=data,
    time_col=time_col,
    event_col=event_col,
    group_col=group_col
)

print("\nLog-rank test results:")
print(f"  Test statistic: {logrank_results['test_statistic']:.4f}")
print(f"  p-value: {logrank_results['p_value']:.4f}")

if logrank_results['p_value'] < 0.05:
    print("  → Survival curves are significantly different")
else:
    print("  → No significant difference between groups")


Log-rank test results:
  Test statistic: 136.3387
  p-value: 0.0000
  → Survival curves are significantly different


In [40]:
# Save log-rank results
with open('outputs/logrank_results.json', 'w') as f:
    json.dump(logrank_results, f, indent=2)

print("✅ Saved: outputs/logrank_results.json")

✅ Saved: outputs/logrank_results.json


In [41]:
# Generate KM plot
plot_km_curves(
    km_models=km_models,
    filename='outputs/km_survival_plot.png',
    title=f'Kaplan-Meier Survival Curves by {group_col}'
)

print("✅ Saved: outputs/km_survival_plot.png")

✅ Saved: outputs/km_survival_plot.png


## 3. Cox Proportional Hazards Model

Fit regression model with ≥3 covariates.

In [42]:
# Select covariates (must have at least 3)
# RADCURE example covariates:
covariates = [
    'Age',                    # Continuous
    'Sex',                 # Categorical
    'Stage',          # Categorical (I, II, III, IV)
    'HPV',           # Categorical (if available)
    'Chemo',           # Binary (Yes/No)
    'Dose'              # Continuous
]

# Verify columns exist and handle encoding
print("Checking covariates...")
available_covariates = [col for col in covariates if col in data.columns]
print(f"Available: {available_covariates}")

# For categorical variables, may need to encode
# Lifelines handles this automatically, but check data types
for col in available_covariates:
    print(f"{col}: {data[col].dtype}, unique values: {data[col].nunique()}")

# Use at least 3 available covariates
covariates = available_covariates[:6]  # Use first 6 if available
print(f"\nUsing {len(covariates)} covariates: {covariates}")

Checking covariates...
Available: ['Age', 'Sex', 'Stage', 'HPV', 'Chemo', 'Dose']
Age: float64, unique values: 544
Sex: object, unique values: 2
Stage: object, unique values: 14
HPV: object, unique values: 2
Chemo: object, unique values: 2
Dose: float64, unique values: 20

Using 6 covariates: ['Age', 'Sex', 'Stage', 'HPV', 'Chemo', 'Dose']


In [43]:
print(available_covariates)
print(covariates)

['Age', 'Sex', 'Stage', 'HPV', 'Chemo', 'Dose']
['Age', 'Sex', 'Stage', 'HPV', 'Chemo', 'Dose']


In [44]:
# Fit Cox model
cox = fit_cox_model(
    data=data,
    time_col=time_col,
    event_col=event_col,
    covariates=covariates
)

print("✅ Cox model fitted")

✅ Cox model fitted


In [45]:
# Extract summary
cox_summary = extract_cox_summary(cox)

print("\nCox Model Summary:")
print(cox_summary)


Cox Model Summary:
            covariate       coef     exp(coef)      se(coef)  coef lower 95%  \
0                 Age   0.031868  1.032381e+00      0.003171        0.025652   
1                Dose  -0.020251  9.799522e-01      0.007752       -0.035445   
2            Sex_Male   0.097110  1.101982e+00      0.077898       -0.055567   
3             Stage_I  -0.367441  6.925038e-01      0.262790       -0.882500   
4            Stage_IB -16.062444  1.057230e-07  18984.912041   -37225.806294   
5            Stage_II   0.131914  1.141011e+00      0.268585       -0.394502   
6           Stage_IIA -11.348491  1.178726e-05    611.830271    -1210.513788   
7           Stage_IIB   2.102176  8.183956e+00      1.030849        0.081748   
8           Stage_III   0.601225  1.824352e+00      0.267304        0.077319   
9          Stage_IIIA   2.588322  1.330742e+01      0.748823        1.120657   
10         Stage_IIIC   2.644326  1.407395e+01      0.749442        1.175446   
11           Stage_I

In [46]:
# Save Cox summary
cox_summary.to_csv('outputs/cox_summary.csv', index=False)
print("✅ Saved: outputs/cox_summary.csv")

✅ Saved: outputs/cox_summary.csv


In [47]:
# Test proportional hazards assumption
ph_test = test_proportional_hazards(
    cox_model=cox,
    data=data,
    time_col=time_col,
    event_col=event_col
)

print("\nProportional Hazards Test:")
for covar, result in ph_test.items():
    status = "✓" if result['p_value'] > 0.05 else "✗"
    print(f"  {status} {covar}: p = {result['p_value']:.4f}")


Proportional Hazards Test:
  ✗ Age: p = 0.0002
  ✓ Chemo_none: p = 0.8654
  ✓ Dose: p = 0.8623
  ✗ HPV_Yes, positive: p = 0.0340
  ✓ Sex_Male: p = 0.0704
  ✓ Stage_I: p = 0.0631
  ✓ Stage_IB: p = 0.9999
  ✓ Stage_II: p = 0.2211
  ✓ Stage_IIA: p = 0.9993
  ✓ Stage_IIB: p = 0.8255
  ✓ Stage_III: p = 0.8014
  ✓ Stage_IIIA: p = 0.6716
  ✓ Stage_IIIC: p = 0.6289
  ✓ Stage_IV: p = 0.8356
  ✓ Stage_IVA: p = 0.5299
  ✓ Stage_IVB: p = 0.0845
  ✓ Stage_IVC: p = 0.9996
  ✓ Stage_X: p = 0.5566


In [48]:
# Save PH test results
with open('outputs/cox_ph_test.json', 'w') as f:
    json.dump(ph_test, f, indent=2)

print("✅ Saved: outputs/cox_ph_test.json")

✅ Saved: outputs/cox_ph_test.json


## 4. Random Survival Forest

Train ensemble model and evaluate performance.

In [49]:
import sys
print(sys.executable)

import sksurv
print(sksurv.__version__)

c:\Users\JAY\AppData\Local\Programs\Python\Python312\python.exe
0.28.0


In [50]:
print(data["Status"].value_counts())

Status
0    2288
1    1058
Name: count, dtype: int64


In [51]:
# Prepare features
from sksurv.util import Surv

# Select features (can be same as Cox covariates or different)
feature_cols = covariates
X = data[feature_cols].copy()

X = pd.get_dummies(X, drop_first=True)  # One-hot encode categorical variables

data["event"] = data["Status"].astype(bool)
data["time"] = data["Length FU"]

# Prepare survival outcome
y = Surv.from_dataframe('event', 'time', data)

print(f"Feature matrix shape: {X.shape}")
print(f"Survival outcome shape: {y.shape}")

Feature matrix shape: (3346, 18)
Survival outcome shape: (3346,)


In [52]:
# Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

Training set: 2342 samples
Test set: 1004 samples


In [53]:
# Fit Random Survival Forest
rsf = fit_random_survival_forest(
    X_train=X_train,
    y_train=y_train,
    n_estimators=100,
    random_state=42
)

print("✅ Random Survival Forest trained")

✅ Random Survival Forest trained


In [54]:
# Compute C-index
c_index = compute_concordance_index(rsf, X_test, y_test)

print(f"\nConcordance Index (C-index): {c_index:.4f}")
print("\nInterpretation:")
if c_index > 0.7:
    print("  → Good predictive performance")
elif c_index > 0.6:
    print("  → Moderate predictive performance")
else:
    print("  → Weak predictive performance")


Concordance Index (C-index): 0.7000

Interpretation:
  → Good predictive performance


In [55]:
# Save metrics
rsf_metrics = {'c_index': float(c_index)}

with open('outputs/rsf_metrics.json', 'w') as f:
    json.dump(rsf_metrics, f, indent=2)

print("✅ Saved: outputs/rsf_metrics.json")

✅ Saved: outputs/rsf_metrics.json


In [56]:
importance = get_feature_importance(
    rsf,
    X_train.columns.tolist()
)

print(importance.head())

    feature  importance
0       Age    0.055556
1      Dose    0.055556
2  Sex_Male    0.055556
3   Stage_I    0.055556
4  Stage_IB    0.055556


In [57]:
# Plot feature importance
plot_feature_importance(
    importance_df=importance,
    filename='outputs/rsf_importance.png',
    top_n=10
)

print("✅ Saved: outputs/rsf_importance.png")

✅ Saved: outputs/rsf_importance.png


## 5. Verify All Outputs

Check that all required files were generated.

In [58]:
required_files = [
    'outputs/km_survival_plot.png',
    'outputs/logrank_results.json',
    'outputs/cox_summary.csv',
    'outputs/cox_ph_test.json',
    'outputs/rsf_importance.png',
    'outputs/rsf_metrics.json',
]

print("Checking required outputs:\n")
all_exist = True
for filepath in required_files:
    exists = Path(filepath).exists()
    status = "✅" if exists else "❌"
    print(f"{status} {filepath}")
    all_exist = all_exist and exists

print("\n" + "="*50)
if all_exist:
    print("✅ ALL OUTPUTS GENERATED")
    print("Ready to push to GitHub!")
else:
    print("❌ MISSING OUTPUTS")
    print("Re-run the cells above to generate missing files.")

Checking required outputs:

✅ outputs/km_survival_plot.png
✅ outputs/logrank_results.json
✅ outputs/cox_summary.csv
✅ outputs/cox_ph_test.json
✅ outputs/rsf_importance.png
✅ outputs/rsf_metrics.json

✅ ALL OUTPUTS GENERATED
Ready to push to GitHub!


## Next Steps

1. ✅ Complete `AI_DISCLOSURE.md`
2. ✅ Run `python validate_submission.py` locally
3. ✅ Push to GitHub
4. ✅ Check that GitHub Actions tests pass
5. ✅ Upload outputs to Blackboard (if required)

Good luck! 🎉